# 02 - ACE Training
Train the localized ACE model on the dataset and log metrics.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.ace_wrapper import ACEWrapper
from src.trainer import BenchmarkTrainer

In [2]:
# Load Data
train_ds = MDTrajectoryDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MDTrajectoryDataset("../data/val.extxyz", cutoff=5.0)
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize ACE model (shared training setup, model-specific architecture)
model = ACEWrapper(
    num_elements=120,
    num_radial=8,
    l_max=2,
    r_cut=5.0,
    hidden_dim=32
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)
print(f"Training on device: {trainer.device}\n")

Training on device: cuda



In [4]:
# Train + held-out test evaluation
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/ace_metrics.csv", index=False)

test_metrics = trainer.test_epoch(test_loader)
ace_test_df = pd.DataFrame([test_metrics])
ace_test_df.to_csv("../data/ace_test_metrics.csv", index=False)

# Save the trained model state
torch.save(model.state_dict(), "../data/ace_model.pth")

metrics_df.head()

Epoch 000 | Time: 3.54s | Train E MAE: 18.85 meV/atom | Train F MAE: 227.85 meV/Å | Val E MAE: 15.20 meV/atom | Val F MAE: 174.72 meV/Å
Epoch 001 | Time: 1.62s | Train E MAE: 12.15 meV/atom | Train F MAE: 102.93 meV/Å | Val E MAE: 3.91 meV/atom | Val F MAE: 76.16 meV/Å
Epoch 002 | Time: 1.01s | Train E MAE: 4.25 meV/atom | Train F MAE: 69.17 meV/Å | Val E MAE: 1.82 meV/atom | Val F MAE: 64.95 meV/Å
Epoch 003 | Time: 0.95s | Train E MAE: 1.80 meV/atom | Train F MAE: 61.97 meV/Å | Val E MAE: 1.42 meV/atom | Val F MAE: 57.58 meV/Å
Epoch 004 | Time: 0.92s | Train E MAE: 6.24 meV/atom | Train F MAE: 53.26 meV/Å | Val E MAE: 9.70 meV/atom | Val F MAE: 47.86 meV/Å
Epoch 005 | Time: 0.92s | Train E MAE: 6.63 meV/atom | Train F MAE: 43.89 meV/Å | Val E MAE: 1.99 meV/atom | Val F MAE: 38.72 meV/Å
Epoch 006 | Time: 0.92s | Train E MAE: 6.27 meV/atom | Train F MAE: 33.64 meV/Å | Val E MAE: 13.98 meV/atom | Val F MAE: 28.10 meV/Å
Epoch 007 | Time: 0.95s | Train E MAE: 11.61 meV/atom | Train F MAE: 

,epoch,loss,e_mae,f_mae,time,val_loss,val_e_mae,val_f_mae
0,0,8.109809,18.854351,227.852840,3.536173,4.807912,15.195015,174.716043
1,1,1.829734,12.147100,102.933500,1.620632,0.882841,3.914083,76.158887
2,2,0.723064,4.251077,69.167023,1.008802,0.638785,1.818653,64.951921
3,3,0.577450,1.802176,61.974616,0.945165,0.502052,1.419304,57.583597
4,4,0.433216,6.237875,53.263623,0.923739,0.352685,9.696251,47.863705


In [5]:
# Check model size (number of parameters)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 6,474
Trainable Parameters: 6,465
